# **Class 1: Learning atomistic properties with machine learning**

The goal of today is to gain hands-on experience with machine learning tools used in atomistic simulations.


❗ To change between CPU and GPU in this notebook:

     Click on the runtime menu ➡ change runtime type ➡ toggle between CPU & T4 GPU

You will probably only need CPU for this class.

# **Install prerequisite packages**

In [ ]:
!pip install quippy-ase
!pip install graph-pes chemiscope
!pip install ipywidgets

# **Part 1: understand, explore, and visualise atomistic data**

During this exercise we will learn how to load atomic structures from an .extxyz file, visualise them, and perform basic operations such as calculating distances and angles between atoms.

You will be asked to write some code to perform these operations, and so you should read the documentation of the packages we will use. The documentation is available at:

*   ASE (Atomic Simulation Environment): https://wiki.fysik.dtu.dk/ase/
*   Load-atoms (package for loading published datasets from repositories) https://jla-gardner.github.io/load-atoms/

We will be using a synthetic dataset for carbon. It contains around 500 carbon structures covering a wide range of densities, temperatures and degrees of (dis)order.

More detail can be found in the [original paper](https://pubs.rsc.org/en/content/articlelanding/2023/dd/d2dd00137c).

**Load data**

In [ ]:
%%bash

if [ ! -f structures_filt.xyz ]; then
    wget https://raw.githubusercontent.com/nfragapane/Atomistic-ML-Tutorial/main/Class-1/structures_filt.xyz -O structures_filt.xyz
fi

In [ ]:
# load dataset
from load_atoms import load_dataset
import ase.io

structures = load_dataset("structures_filt.xyz")
print(len(structures))

In [ ]:
# explore contents of dataset
structures

In [ ]:
# visualise some of the structures in the dataset by changing the index
from load_atoms import view

idx = 30
view(structures[idx], show_bonds=True)

In [ ]:
# see more info about each structure like this
structures[0].info

In [ ]:
# write the structures to a file and inspect its content
# you can use the write function from ase.io
# it will be saved in the file tab to the left

...

**Generate descriptors**

Descriptors are a way to represent atomic structures in a way that is suitable for machine learning. They are typically based on the positions of atoms and their chemical environments. In this exercise, you should implement a simple descriptor based on the distances between atoms, the angle between atoms and also the coordination number. Refer to the ASE documentation for functions to help calculate these properties (or if you are feeling adventurous, you can implement them yourself)!



Some questions to consider:

*   What are the differences in distances between structures with different densities?
*   What are the differences in angles between structures with different densities?
*   What are the coordination numbers in structures with different densities? What happens when you change the cutoff radius? Is it possible to find a cutoff radius which encompasses first neighbour bonding information?
*   Can you find patterns across the different structures and the corresponding descriptors?
*   Look at the size of the descriptors generated. How much more data are you generating through angular compared to distance descriptors? What body order descriptors are you generating?

In [ ]:
from ase.neighborlist import neighbor_list

In [ ]:
structure_0 = structures[0]

i, j, d = neighbor_list("ijd", structure_0, cutoff=3.7)
print(i)
print(j)
print(d)

In [ ]:
import numpy as np

coordination_num = np.bincount(i)

In [ ]:
coordination_num

In [ ]:
import matplotlib.pyplot as plt

plt.hist(coordination_num, bins=20)

We can build a representation of data through the radial distribution function (RDF) and the angular distribution function (ADF).

The RDF is a measure of the probability of finding an atom at a certain distance from another atom, while the ADF is a measure of the probability of finding an atom at a certain angle with respect to another atom.

These functions are useful for understanding the local structure of materials and can be used to identify different phases or structures and can be generated experimentally through X-ray diffraction or neutron scattering experiments.

In [ ]:
# plot the radial distribution function – look in the ASE docs for how to do this:
# https://wiki.fysik.dtu.dk/ase/ase/neighborlist.html#ase.neighborlist.neighbor_list

In [ ]:
# plot the angular distribution function – look at the ASE docs for how to do this:
# https://wiki.fysik.dtu.dk/ase/ase/geometry.html#ase.geometry.analysis.Analysis.get_angles

from ase.geometry.analysis import Analysis

analysis = Analysis(structure_0)
CCCAngles = analysis.get_angles("C", "C", "C", unique=True)
CCCAngleValues = analysis.get_values(CCCAngles)

plt.hist(CCCAngleValues, bins=40)

✅ Two- and three-body descriptors

✅ Building a representation of data through the radial distribution function (RDF) and the angular distribution function (ADF)

# **Part 2: exploring the structural space of carbon using SOAP**

We now move to many-body descriptors, which provide a more complex representation of atomic structures. Unlike simpler pairwise and angular descriptors, many-body descriptors account for interactions among more atoms, capturing richer structural information.

One widely used many-body descriptor is the Smooth Overlap of Atomic Positions (SOAP). SOAP is a powerful and flexible tool for characterising local atomic environments, enabling accurate predictions in machine learning models and aiding in the identification of structural motifs or phases in materials.

SOAP represents the local atomic environment around each atom as a continuous atomic density, constructed by placing Gaussian functions on neighboring atoms. This smooth density is then expanded in a basis of radial functions and spherical harmonics. We can compare atomic environments by computing the similarity between these densities, using an inner product that is invariant to rotations and permutations.

However, since SOAP descriptors are high dimensional, visualising data can become difficult. To overcome this, we can use dimensionality reduction techniques such as principal component analysis (PCA) to reduce the dimensionality of the SOAP descriptors. PCA is a statistical technique that transforms the data into a new coordinate system, where the greatest variance is captured in the first few dimensions. This allows us to visualise the data in a lower-dimensional space while retaining most of the information (hopefully). We collaquially refer to these as "SOAP maps".

**Atomic-scale representation (local environments)**

In [ ]:
# Lets build the SOAP descriptor using quippy, a package for calculating atomic-scale descriptors. The documentation is here: https://libatoms.github.io/QUIP/
from quippy.descriptors import Descriptor

desc = Descriptor("soap cutoff=3.7 n_max=4 l_max=4 atom_sigma=0.5") # Play around with the parameters to see how they affect the descriptor. cutoff is the cutoff radius, n_max is the maximum number of radial basis functions, l_max is the maximum number of angular basis functions, and atom_sigma is the width of the Gaussian used to smooth the atomic density.
soaps = desc.calc(structure_0)["data"]
soaps.shape

In [ ]:
# Let's write a function to do dimensionality reduction for us
# Add other methods to the below function

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


def do_analysis(data, method, **kwargs):
    """
    Function to perform a dimensionality reduction analysis on the
    descriptors.
    """

    # scale the data.
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data)

    # perform the analysis.
    if method == "pca":
        pca = PCA(n_components=2)
        pca.fit(scaled_data)
        x_pca = pca.transform(scaled_data)
        # print(scaled_data.shape)
        # print(x_pca.shape)
        return x_pca

    else:
        print("Error: method not recognised.")
        return None

In [ ]:
pca_data = do_analysis(soaps, "pca")

In [ ]:
import chemiscope

properties = {
    "PCA": {
        "target": "atom",
        "values": pca_data,
        "description": "PCA of per-atom representation of the structures",
    },
    "coordination_num": {
        "target": "atom",
        "values": coordination_num,
        "description": "Coordination number of each atom",
    },
    "local_energy": {
        "target": "atom",
        "values": structure_0.arrays["local_energies"],
        "description": "Local energies predicted with C-GAP-17",
    },
}

ats_envs = chemiscope.all_atomic_environments(
    [structure_0],
    cutoff=3.7
)

# Write chemiscope dataset
chemiscope.write_input(
    path="chemiscope_data.json.gz",
    structures=[structure_0],
    properties=properties,
    environments=ats_envs,
)

In [ ]:
# download the chemiscope data
from google.colab import files
files.download("chemiscope_data.json.gz")

Now visit the [chemiscope website](https://chemiscope.org/), and upload the `chemiscope_data.json.gz` to visualise.

Click on the `atom` button at the bottom of the right screen to see more atom-level information.



In [ ]:
# try again but for structures with higher densities ...

**Structure-level descriptors (global properties)**

In [ ]:
from quippy.descriptors import Descriptor

desc = Descriptor("soap cutoff=3.7 n_max=4 l_max=4 atom_sigma=0.5 average=T")
soaps = np.array([desc.calc(s)["data"] for s in structures])
soaps.shape
soaps = soaps.reshape(soaps.shape[0], -1)

In [ ]:
pca_data = do_analysis(soaps, "pca")

In [ ]:
import chemiscope

properties = {
    "PCA": {
        "target": "structure",
        "values": pca_data,
        "description": "PCA of per-atom representation of the structures",
    },
    "density": {
        "target": "structure",
        "values": [s.info["density"] for s in structures],
        "description": "Density of the structure",
    },
    "total_energy": {
        "target": "structure",
        "values": [s.info["energy"] for s in structures],
        "description": "Total energies predicted with C-GAP-17",
    },
    "anneal_T": {
        "target": "structure",
        "values": [s.info["anneal_T"] for s in structures],
        "description": "Annealing temperature of the structure",
    },
}

# Write chemiscope dataset to file
chemiscope.write_input(
    path="chemiscope_all_structures.json.gz",
    structures=structures,
    properties=properties,
)

# download the file and view in the website again
files.download("chemiscope_all_structures.json.gz")

✅ Many-body descriptors

✅ Visualising data with PCA maps

# **Part 3: supervised learning- predicting local energies**

We have now generated a large number of descriptors and we can use these to predict the local energy of the atomic structures. We will use a simple linear regression model to predict the local energy of the atomic structures based on the descriptors we have generated. We will also use cross-validation to evaluate the performance of the model and to ensure that it is not overfitting to the training data.

We will use the scikit-learn package for this. The documentation is available at: https://scikit-learn.org/stable/

In [ ]:
# load structures and split the data into training, validation and test
structures = load_dataset("structures_filt.xyz")
train, val, test = structures.random_split([0.8, 0.1, 0.1], seed=42)

In [ ]:
# get the target labels
energies_train = train.arrays["local_energies"]
energies_val = val.arrays["local_energies"]
energies_test = test.arrays["local_energies"]

In [ ]:
# generate the SOAP descriptors for the training, validation, and test sets

desc = Descriptor("soap cutoff=3.7 n_max=4 l_max=4 atom_sigma=0.5")
soaps_train = np.array([desc.calc(s)["data"] for s in train])
soaps_val = np.array([desc.calc(s)["data"] for s in val])
soaps_test = np.array([desc.calc(s)["data"] for s in test])

**Linear regression**

You can implement your own linear model or use scikit-learn's implementation; familiarise yourself with how the model is initialised, trained, and validated/tested

The descriptors are reshaped so that the first dimension matches that of the labels.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(soaps_train.reshape(-1, soaps_train.shape[-1]), energies_train)

In [ ]:
energies_train_pred = model.predict(soaps_train.reshape(-1, soaps_train.shape[-1]))
energies_val_pred = model.predict(soaps_val.reshape(-1, soaps_val.shape[-1]))
energies_test_pred = model.predict(soaps_test.reshape(-1, soaps_test.shape[-1]))

In [ ]:
plt.scatter(energies_train, energies_train_pred, s=4, label="reference")
plt.scatter(energies_test, energies_test_pred, s=4, label="ML")
plt.axline((energies_train[0], energies_train[0]), slope=1, color="red", linestyle="--")

plt.legend()

In [ ]:
# evaluate the model's performance by computing the mean absolute error (MAE) and the root mean square error (RMSE) on the test set

**Ridge regression**

Ridge regression differs from the linear regression by the introduction of the regularisation term, noted alpha in scikit-learn.

In [ ]:
from sklearn.linear_model import Ridge

alpha = ...  # experiment with different values of alpha
model = Ridge(alpha=alpha)
model.fit(soaps_train.reshape(-1, soaps_train.shape[-1]), energies_train)

In [ ]:
train_pred = model.predict(soaps_train.reshape(-1, soaps_train.shape[-1]))
val_pred = model.predict(soaps_val.reshape(-1, soaps_val.shape[-1]))
test_pred = model.predict(soaps_test.reshape(-1, soaps_test.shape[-1]))

In [ ]:
plt.scatter(energies_train, energies_train_pred, s=4, label="reference")
plt.scatter(energies_test, energies_test_pred, s=4, label="ML")
plt.axline((energies_train[0], energies_train[0]), slope=1, color="red", linestyle="--")

plt.legend()

In [ ]:
# using the validation set, find the optimal value of the regularisation and evaluate the performance metrics of this model on the test set

✅ Train and evaluate simple supervised models for energy prediction.

**Optional extensions**

* Try the neural network model from scikit-learn
* Implement a simple kernel model (or Gaussian Process Regression model) as detailed in the GPR review; you can also use kernel ridge regression from scikit-learn.

# **Part 4: atomic property prediction (optional)**

We can predict other atomic-level properties, rather than just the local energies, with a similar workflow.

Tasks:


*   Modify the pipeline above to predict chemical shifts on a per-atom basis.
*   Train and evaluate your model.


Prompts:


*   Would you use a different descriptor for this property?







In [ ]:
# write your new pipeline here...

%%bash

if [ ! -f aSiO2_chem_shift.xyz ]; then
    wget https://raw.githubusercontent.com/nfragapane/Atomistic-ML-Tutorial/main/Class-1/aSiO2_chem_shift.xyz -O aSiO2_chem_shift.xyz
fi

✅ Train and evaluate models for chemical shifts.

# **Extension tasks**

For those who finish early or want to explore further, here are some extension tasks:

*   Use your model to make predictions on unseen or extrapolated structures.
*   Generate and interpret learning curves to understand data requirements.
*   Train models to predict atomic forces or stress tensors.
*   Visualise feature importance or atomic contributions.